# Social Media Engagement Analyzer with Hypothesis Testing
**Student:** Swara Chintawar  
**Roll No.:** 18  
**Enrollment No.:** 24RBU/UG/CS/02/000162  
**Section:** B1  
**Dashboard:** Streamlit

## Project overview
This notebook contains the full project workflow: setup, reproducible demo dataset creation, data preparation, exploratory analysis, platform hypothesis testing using one-way ANOVA, machine-learning model comparison, and creation/launch of an interactive Streamlit dashboard.

> The built-in dataset is synthetic demo data. Upload your real CSV in the dashboard for real-data results. Required columns: `Platform`, `Content_Type`, `Date`, `Impressions`, `Likes`, `Comments`, `Shares`. `Post_ID` is optional.

## 1. Install and import required libraries

In [ ]:
%pip -q install streamlit pandas numpy plotly scipy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print("Libraries imported successfully.")

## 2. Create a reproducible demo dataset

In [ ]:
rng = np.random.default_rng(42)
n = 150
impressions = rng.integers(500, 15000, n)
likes = np.maximum(0, (impressions * rng.uniform(.015, .09, n)).astype(int))
comments = np.maximum(0, (likes * rng.uniform(.02, .18, n)).astype(int))
shares = np.maximum(0, (likes * rng.uniform(.01, .13, n)).astype(int))

demo_df = pd.DataFrame({
    "Post_ID": [f"P{i:03d}" for i in range(1, n + 1)],
    "Platform": rng.choice(["Instagram", "Facebook", "YouTube", "X"], n),
    "Content_Type": rng.choice(["Image", "Video", "Reel", "Text"], n),
    "Date": pd.date_range("2026-01-01", periods=n).strftime("%Y-%m-%d"),
    "Impressions": impressions,
    "Likes": likes,
    "Comments": comments,
    "Shares": shares
})
demo_df.to_csv("social_media_demo.csv", index=False)
print("Demo dataset created:", len(demo_df), "posts")
display(demo_df.head())

## 3. Clean data and calculate engagement metrics

In [ ]:
REQUIRED = ["Platform", "Content_Type", "Date", "Impressions", "Likes", "Comments", "Shares"]
analysis_df = demo_df.copy()
for col in ["Impressions", "Likes", "Comments", "Shares"]:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")
analysis_df = analysis_df.dropna(subset=REQUIRED)
analysis_df = analysis_df[(analysis_df[["Impressions", "Likes", "Comments", "Shares"]] >= 0).all(axis=1)]
analysis_df = analysis_df[analysis_df["Impressions"] > 0].copy()
analysis_df["Engagements"] = analysis_df[["Likes", "Comments", "Shares"]].sum(axis=1)
analysis_df["Engagement_Rate_%"] = 100 * analysis_df["Engagements"] / analysis_df["Impressions"]
print("Valid rows:", len(analysis_df))
print("Missing values after cleaning:")
display(analysis_df.isna().sum().to_frame("Missing values"))
display(analysis_df.select_dtypes(include="number").describe().round(2))

## 4. Exploratory data analysis: platform and content type

In [ ]:
platform_comparison = analysis_df.groupby("Platform").agg(
    Posts=("Post_ID", "count"),
    Total_Impressions=("Impressions", "sum"),
    Total_Engagements=("Engagements", "sum"),
    Mean_Engagement_Rate=("Engagement_Rate_%", "mean")
).reset_index()
platform_comparison["Weighted_Engagement_Rate_%"] = 100 * platform_comparison["Total_Engagements"] / platform_comparison["Total_Impressions"]
print("Platform comparison")
display(platform_comparison.sort_values("Weighted_Engagement_Rate_%", ascending=False).round(2))

content_comparison = analysis_df.groupby("Content_Type").agg(
    Posts=("Post_ID", "count"),
    Total_Impressions=("Impressions", "sum"),
    Total_Engagements=("Engagements", "sum"),
    Mean_Engagement_Rate=("Engagement_Rate_%", "mean")
).reset_index()
content_comparison["Weighted_Engagement_Rate_%"] = 100 * content_comparison["Total_Engagements"] / content_comparison["Total_Impressions"]
print("Content type comparison")
display(content_comparison.sort_values("Weighted_Engagement_Rate_%", ascending=False).round(2))

## 5. Hypothesis testing using one-way ANOVA

In [ ]:
# H0: Mean engagement rate is equal across platforms.
# H1: At least one platform has a different mean engagement rate.
alpha = 0.05
platform_groups = [g["Engagement_Rate_%"].to_numpy() for _, g in analysis_df.groupby("Platform") if len(g) > 1]
if len(platform_groups) >= 2:
    f_stat, p_value = stats.f_oneway(*platform_groups)
    print(f"F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value:.6f}")
    print("Decision:", "Reject H0; a statistically significant difference was detected." if p_value < alpha else "Fail to reject H0; no statistically significant difference was detected.")
else:
    print("ANOVA requires at least two platform groups with more than one observation each.")

## 6. Machine-learning model comparison

In [ ]:
X = analysis_df[["Impressions", "Platform", "Content_Type"]]
y = analysis_df["Engagement_Rate_%"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), ["Impressions"]),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), ["Platform", "Content_Type"])
])
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}
results = []
for name, estimator in models.items():
    model = Pipeline([("preprocess", preprocess), ("model", estimator)])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"Model": name, "MAE (lower is better)": mean_absolute_error(y_test, pred), "RMSE (lower is better)": mean_squared_error(y_test, pred) ** 0.5, "R2 (higher is better)": r2_score(y_test, pred)})
model_comparison = pd.DataFrame(results).sort_values(["MAE (lower is better)", "RMSE (lower is better)"]).reset_index(drop=True)
display(model_comparison.round(4))
print("Best model on this demo-data split:", model_comparison.iloc[0]["Model"])
print("Note: results may change when using real data.")

## 7. Generate the Streamlit dashboard application

Run this cell to write `app.py` into the current Colab working directory. The dashboard includes interactive filters, metrics, charts, ANOVA hypothesis testing, model comparison, data tables, CSV upload and download.

In [ ]:
app_code = r'''import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

st.set_page_config(page_title="Social Media Engagement Analyzer", page_icon="📱", layout="wide")

REQUIRED = ["Platform", "Content_Type", "Date", "Impressions", "Likes", "Comments", "Shares"]

@st.cache_data
def demo_data():
    rng = np.random.default_rng(42)
    n = 150
    impressions = rng.integers(500, 15000, n)
    likes = np.maximum(0, (impressions * rng.uniform(.015, .09, n)).astype(int))
    comments = np.maximum(0, (likes * rng.uniform(.02, .18, n)).astype(int))
    shares = np.maximum(0, (likes * rng.uniform(.01, .13, n)).astype(int))
    return pd.DataFrame({
        "Post_ID": [f"P{i:03d}" for i in range(1, n + 1)],
        "Platform": rng.choice(["Instagram", "Facebook", "YouTube", "X"], n),
        "Content_Type": rng.choice(["Image", "Video", "Reel", "Text"], n),
        "Date": pd.date_range("2026-01-01", periods=n).strftime("%Y-%m-%d"),
        "Impressions": impressions, "Likes": likes, "Comments": comments, "Shares": shares
    })

def prepare_data(df):
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError("CSV is missing required columns: " + ", ".join(missing))
    df = df.copy()
    for c in ["Impressions", "Likes", "Comments", "Shares"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=REQUIRED)
    df = df[(df[["Impressions", "Likes", "Comments", "Shares"]] >= 0).all(axis=1)]
    df = df[df["Impressions"] > 0].copy()
    df["Date"] = df["Date"].astype(str)
    df["Engagements"] = df[["Likes", "Comments", "Shares"]].sum(axis=1)
    df["Engagement_Rate_%"] = 100 * df["Engagements"] / df["Impressions"]
    if df.empty:
        raise ValueError("No valid rows found. Check that numeric columns contain valid non-negative values and impressions are greater than zero.")
    if "Post_ID" not in df.columns:
        df["Post_ID"] = [f"Post {i+1}" for i in range(len(df))]
    return df

st.title("📱 Social Media Engagement Analyzer")
st.caption("Analyze post performance, compare platforms and content types, and test whether engagement rates differ statistically.")

with st.sidebar:
    st.header("⚙️ Dashboard Controls")
    uploaded = st.file_uploader("Upload your social media CSV (optional)", type=["csv"])
    st.caption("Without an upload, the dashboard uses 150 reproducible synthetic demo posts.")
    st.divider()
    st.subheader("Filters")
    min_rate = st.slider("Minimum engagement rate (%)", min_value=0.0, max_value=30.0, value=0.0, step=0.5)
    sort_by = st.selectbox("Sort posts by", ["Highest engagement rate", "Most likes", "Most impressions"])
    st.divider()
    st.caption("Required CSV columns: Platform, Content_Type, Date, Impressions, Likes, Comments, Shares")

try:
    raw_df = pd.read_csv(uploaded) if uploaded else demo_data()
    df = prepare_data(raw_df)
except Exception as e:
    st.error(str(e))
    st.stop()

platforms = ["All"] + sorted(df["Platform"].astype(str).unique().tolist())
content_types = ["All"] + sorted(df["Content_Type"].astype(str).unique().tolist())
f1, f2 = st.columns(2)
with f1:
    selected_platform = st.selectbox("Platform", platforms)
with f2:
    selected_content = st.selectbox("Content type", content_types)

filtered = df.copy()
if selected_platform != "All":
    filtered = filtered[filtered["Platform"].astype(str) == selected_platform]
if selected_content != "All":
    filtered = filtered[filtered["Content_Type"].astype(str) == selected_content]
filtered = filtered[filtered["Engagement_Rate_%"] >= min_rate]
if filtered.empty:
    st.warning("No posts match the selected filters. Please change the filters or lower the minimum engagement rate.")
    st.stop()

sort_map = {"Highest engagement rate": "Engagement_Rate_%", "Most likes": "Likes", "Most impressions": "Impressions"}
filtered = filtered.sort_values(sort_map[sort_by], ascending=False)
total_impressions = int(filtered["Impressions"].sum())
total_engagements = int(filtered["Engagements"].sum())
overall_rate = total_engagements / total_impressions * 100 if total_impressions else 0

st.subheader("📊 Performance Overview")
k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("Posts analyzed", f"{len(filtered):,}")
k2.metric("Total impressions", f"{total_impressions:,}")
k3.metric("Total likes", f"{int(filtered['Likes'].sum()):,}")
k4.metric("Total engagements", f"{total_engagements:,}")
k5.metric("Engagement rate", f"{overall_rate:.2f}%")
st.caption(f"Top post by {sort_by.lower()}: **{filtered.iloc[0]['Post_ID']}**")

st.divider()
st.subheader("📈 Engagement Insights")
chart1, chart2 = st.columns(2)
with chart1:
    platform_group = filtered.groupby("Platform", as_index=False)[["Engagements", "Impressions"]].sum()
    platform_group["Engagement Rate (%)"] = 100 * platform_group["Engagements"] / platform_group["Impressions"]
    fig = px.bar(platform_group.sort_values("Engagement Rate (%)", ascending=False), x="Platform", y="Engagement Rate (%)", color="Platform", title="Engagement rate by platform", text_auto=".2f")
    fig.update_layout(template="plotly_white", showlegend=False)
    st.plotly_chart(fig, use_container_width=True)
with chart2:
    content_group = filtered.groupby("Content_Type", as_index=False)[["Engagements", "Impressions"]].sum()
    content_group["Engagement Rate (%)"] = 100 * content_group["Engagements"] / content_group["Impressions"]
    fig = px.bar(content_group.sort_values("Engagement Rate (%)", ascending=False), x="Content_Type", y="Engagement Rate (%)", color="Content_Type", title="Engagement rate by content type", text_auto=".2f")
    fig.update_layout(template="plotly_white", showlegend=False)
    st.plotly_chart(fig, use_container_width=True)

chart3, chart4 = st.columns(2)
with chart3:
    daily = filtered.copy()
    daily["Date"] = pd.to_datetime(daily["Date"], errors="coerce")
    daily = daily.dropna(subset=["Date"]).groupby("Date", as_index=False)[["Engagements", "Impressions"]].sum()
    if not daily.empty:
        daily["Engagement Rate (%)"] = 100 * daily["Engagements"] / daily["Impressions"]
        fig = px.line(daily.sort_values("Date"), x="Date", y="Engagement Rate (%)", markers=True, title="Engagement rate over time")
        fig.update_layout(template="plotly_white")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Dates could not be interpreted for the trend chart.")
with chart4:
    fig = px.scatter(filtered, x="Likes", y="Comments", color="Platform", size="Shares", hover_data=["Post_ID", "Content_Type"], title="Likes vs comments")
    fig.update_layout(template="plotly_white")
    st.plotly_chart(fig, use_container_width=True)

st.divider()
st.subheader("🧪 Hypothesis Testing: One-way ANOVA")
st.markdown("**H₀:** Mean engagement rate is equal across platforms.  \n**H₁:** At least one platform has a different mean engagement rate.  \n**Significance level:** 0.05")
analysis_groups = [g["Engagement_Rate_%"].to_numpy() for _, g in filtered.groupby("Platform") if len(g) > 1]
if len(analysis_groups) >= 2:
    f_stat, p_value = stats.f_oneway(*analysis_groups)
    a1, a2, a3 = st.columns(3)
    a1.metric("F-statistic", f"{f_stat:.4f}")
    a2.metric("P-value", f"{p_value:.4f}")
    a3.metric("Decision", "Reject H₀" if p_value < 0.05 else "Fail to reject H₀")
    if p_value < 0.05:
        st.success("The p-value is below 0.05. There is statistically significant evidence that at least one platform's mean engagement rate differs.")
    else:
        st.info("The p-value is at least 0.05. The analysis did not detect a statistically significant difference in mean engagement rates across platforms.")
else:
    st.warning("ANOVA needs at least two platform groups with more than one valid post each. Select All platforms or upload more data.")

st.subheader("Platform Comparison")
platform_comparison = filtered.groupby("Platform").agg(Posts=("Post_ID", "count"), Total_Impressions=("Impressions", "sum"), Total_Engagements=("Engagements", "sum"), Mean_Engagement_Rate=("Engagement_Rate_%", "mean")).reset_index()
platform_comparison["Weighted_Engagement_Rate_%"] = 100 * platform_comparison["Total_Engagements"] / platform_comparison["Total_Impressions"]
st.dataframe(platform_comparison.round(2), use_container_width=True, hide_index=True)

st.subheader("Content Type Comparison")
content_comparison = filtered.groupby("Content_Type").agg(Posts=("Post_ID", "count"), Total_Impressions=("Impressions", "sum"), Total_Engagements=("Engagements", "sum"), Mean_Engagement_Rate=("Engagement_Rate_%", "mean")).reset_index()
content_comparison["Weighted_Engagement_Rate_%"] = 100 * content_comparison["Total_Engagements"] / content_comparison["Total_Impressions"]
st.dataframe(content_comparison.round(2), use_container_width=True, hide_index=True)

st.divider()
st.subheader("🤖 Machine Learning Model Comparison")
st.caption("Models predict engagement rate using impressions, platform, and content type. The scores depend on the uploaded/demo dataset.")
if len(df) >= 10 and df["Platform"].nunique() >= 1 and df["Content_Type"].nunique() >= 1:
    X = df[["Impressions", "Platform", "Content_Type"]]
    y = df["Engagement_Rate_%"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    numeric_features = ["Impressions"]
    categorical_features = ["Platform", "Content_Type"]
    preprocess = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
    ])
    models = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42)
    }
    results = []
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        results.append({"Model": name, "MAE (lower is better)": mean_absolute_error(y_test, pred), "RMSE (lower is better)": mean_squared_error(y_test, pred) ** 0.5, "R² (higher is better)": r2_score(y_test, pred)})
    model_results = pd.DataFrame(results).sort_values(["MAE (lower is better)", "RMSE (lower is better)"]).reset_index(drop=True)
    best = model_results.iloc[0]
    m1, m2, m3 = st.columns(3)
    m1.metric("Best model", best["Model"])
    m2.metric("Best MAE", f"{best['MAE (lower is better)']:.4f}")
    m3.metric("Best RMSE", f"{best['RMSE (lower is better)']:.4f}")
    st.dataframe(model_results.round(4), use_container_width=True, hide_index=True)
    st.caption("MAE and RMSE should be lower; R² is generally better when higher. These are hold-out test results, not a guarantee of real-world performance.")
else:
    st.warning("At least 10 valid posts are recommended for model comparison.")

st.divider()
st.subheader("🗂️ Posts Table")
st.dataframe(filtered.head(50).round(2), use_container_width=True, hide_index=True)
csv = filtered.to_csv(index=False).encode("utf-8")
st.download_button("⬇️ Download filtered results as CSV", data=csv, file_name="social_media_analysis_results.csv", mime="text/csv")
st.caption("Project by Swara Chintawar | Roll No. 18 | Section B1")
'''
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("Created app.py")
print("You can download app.py from the Colab Files panel, or run it in the next step.")

## 8. Launch the Streamlit dashboard in Google Colab

This launch cell uses a Cloudflare HTTPS tunnel instead of LocalTunnel, which can fail to load Streamlit JavaScript components. Run it after the `app.py` generation cell. Keep it running while using the dashboard.


In [ ]:
# Stop any old Streamlit process and start the dashboard in the background.
!pkill -f "streamlit run app.py" || true
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &

# Download Cloudflare Tunnel (avoids LocalTunnel's flaky asset delivery).
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Print a public HTTPS URL. Keep this cell running while using the dashboard.
!./cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate


### If the dashboard does not open
- Wait until the launch cell prints a `https://...trycloudflare.com` URL, then open that new URL.
- Keep the launch cell running; stopping it closes the public tunnel.
- If the URL opens but widgets do not load, refresh the page with **Ctrl + Shift + R**.
- If the tunnel URL is not printed, inspect the output of the launch cell and the `streamlit.log` file.
